# 29 — Text Normalization for Resumes
**Goal:** Domain-specific cleaning for messy resume text.

Everything Ch. 26–28 extracted is raw: bullet glyphs, ALL-CAPS headers, abbreviations, pipes and slashes splitting skill lists, tabs left over from PDF column reconstruction. Normalization is the layer that turns that mess into predictable, NLP-ready text — without losing the content the downstream stages need.

**Why it matters for resumes / ATS:** an ATS matches keywords, and matching fails on formatting. `Python|NLP/TensorFlow` will not hit a "Python" keyword index as reliably as `Python, NLP, TensorFlow`. Normalizing separators, expanding abbreviations, and collapsing whitespace is cheap and improves every later stage — tokenization, section detection (Ch. 32), and skill matching.

## 1. Resume-Specific Challenges

General-purpose text cleaning is not enough — resumes have a distinctive noise profile: decorative bullet glyphs, abbreviations (`Sr.`, `Engr.`, `w/`), ALL-CAPS section headers, and skill lists whose separators are punctuation soup (`|`, `/`, `.`). A normalizer has to know these patterns and decide what to do with each.

**What the code does:** prints the five issue classes as a checklist — bullet symbols, abbreviations, all-caps headers, skill-list separators, and mixed separators.

**Expected output:** the `Resume text issues:` header followed by the five bullets. The deeper point: two of these classes are *noise* (bullets, whitespace) and two are *signal* — ALL-CAPS headers are the section map Ch. 32 will read, and abbreviations expand into the canonical terms an ATS index expects. Normalization must tell them apart.

In [ ]:
issues = [
    "Bullet symbols: bullet, hyphen, arrow, diamond, triangle",
    "Abbreviations: Sr., Engr., w/, exp., yrs, &",
    "All-caps headers: PROFESSIONAL SUMMARY, SKILLS",
    "Skill lists: Python|NLP/TensorFlow, PyTorch . AWS",
    "Mixed separators: commas, pipes, slashes, bullets",
]
print("Resume text issues:")
for i in issues: print(f"  - {i}")

## 2. Resume Normalizer

`ResumeNormalizer` chains five transforms in a fixed order. The first step is the subtle one: `unicodedata.normalize("NFKD", ...)` followed by an ASCII `encode(..., "ignore")` — this decomposes accented characters and drops everything that still is not ASCII. It strips diacritics (`résumé` → `resume`) and smart punctuation, but it is lossy for genuinely non-Latin text.

**What the code does, step by step:**
- NFKD normalize + ASCII-ignore encode — removes accents and non-ASCII glyphs
- Bullet regex — maps `• ‣ ● ◘ ◙ ➜` to `>`; but the order defeats it: bullets are *already gone* by this step (dropped by the ASCII pass), so the substitution is dead code as written
- `\t|\r` → space; `\n{3,}` → `\n\n`; ` {2,}` → ` `; final `strip()`

**Expected (verified):** the cell as stored has a syntax error — the last line `print(f"After:  {repr(n.normalize(sample))}` is missing its closing `")`, so running it as-is raises `SyntaxError: unterminated f-string literal`. With that typo fixed, `Before:` shows the raw sample with `•` bullets and a tab, and `After:` is `'Srivatsa Gorti\n Python specialist\n NLP engineer\n TensorFlow expert'` — bullets silently removed (leaving a stray space where each was), tab collapsed to a space, double spaces gone. Lesson: transform order is part of the design — the ASCII pass must come *after* any mapping you want to survive it, and an unterminated string at the end of a pipeline cell is exactly the failure that makes a notebook look fine until someone runs it.

In [ ]:
import re, unicodedata

class ResumeNormalizer:
    def normalize(self, text):
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", "ignore").decode()
        text = re.sub(r"[\u2022\u2023\u25cf\u25d8\u25c9\u279c]", ">", text)
        text = re.sub(r"\t|\r", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r" {2,}", " ", text)
        return text.strip()

n = ResumeNormalizer()
sample = "Srivatsa Gorti\n\u2022 Python specialist\n\u2022 NLP engineer\n\tTensorFlow expert"
print(f"Before: {repr(sample)}")
print(f"After:  {repr(n.normalize(sample))}

## 3. Abbreviation Expansion

Abbreviations are a vocabulary problem: `Sr.`, `Engr.`, `w/`, `yrs` all need expanding to canonical forms so keyword matching and later stages see one spelling. This cell builds a dictionary-driven expander with a single case-insensitive regex pass over boundary-delimited alternatives.

**What the code does:**
- Compiles the pattern from the map keys using `re.escape` and `re.IGNORECASE`
- Substitutes via a lambda that looks up the lowercased match in `abbrev_map`

**Expected (verified):** both test lines come back **unchanged**. The stored pattern is `r"\\b(sr\.|...|yrs|yr)\\b"` — note the doubled backslash. In a raw string, `\b` is the word-boundary assertion, but `\\b` compiles to a regex that matches a *literal* backslash followed by `b`; no resume text contains that, so no substitution ever fires and the expander silently does nothing. (Even with the intended single `\b`, only word-final keys such as `yrs` would match — abbreviations ending in `.` or `/`, and `&`, can never satisfy the trailing boundary, since `\b` needs a word character after the punctuation.) Lesson: raw strings make backslashes literal, so `r"\\b"` is not a boundary — an escape-doubling pitfall that also shows why you test the exact stored code, not the intended code.

In [ ]:
abbrev_map = {
    "sr.": "senior", "jr.": "junior", "dr.": "doctor",
    "engr.": "engineer", "mgr.": "manager", "dir.": "director",
    "w/": "with", "&": "and",
    "exp.": "experience", "yrs": "years", "yr": "year",
}
def expand(text):
    pat = re.compile(r"\\b(" + "|".join(re.escape(k) for k in abbrev_map) + r")\\b", re.IGNORECASE)
    return pat.sub(lambda m: abbrev_map[m.group(0).lower()], text)

print(expand("Sr. ML Engr. w/ 10+ yrs exp."))
print(expand("B.Tech CSE & M.S. Data Science"))

## Summary: Resume normalization must preserve content while removing noise. Light touch is better.

**Normalize light — preserve signal, remove noise.** Collapse whitespace and unify separators, expand abbreviations to canonical terms, and strip decorative glyphs; but do not lowercase everything or crush ALL-CAPS headers, because those carry the section structure Ch. 32's section detection depends on. Over-aggressive cleaning (stemming, stopword removal) belongs to a later stage, not this one.

The output of this chapter — clean, predictable text — is the input contract for everything that follows: language detection (Ch. 30) decides which NLP model sees it, and section detection (Ch. 32) finally rebuilds the structured profile.